# Can Barcoding be clinically significant?

Aims:
* Use V2 OCT ART Volume Scans from HEYEX and extract the central scan via `n//2`
* Standardization Functions: Flattening w.r.t. segmentation layer, Intensity normalization, both can be turned off
* Save to a processed file
* Go to ImageJ and mark with a horizontal line, with a parameter for depth
* Plot and save the profile, and extract the intensity series


## Data Loading, Imports, and Setup

In [ ]:
# ============================================================
# Create kernel (run once) and install dependencies
# ============================================================

# Uncomment the following lines the first time only.

# %pip install --upgrade pip
# %pip install ipykernel
# %pip install numpy pandas matplotlib opencv-python pyimagej eyepy tifffile tqdm

# import sys
# !{sys.executable} -m ipykernel install --user \
#      --name barcoding-amd \
#      --display-name "Python (barcoding-amd)"

# Restart the kernel after running once.

In [1]:
from pathlib import Path
import cv2
import imagej
import eyepy as ep
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" 
RAW_DIR = DATA_DIR / "heyex" / "meta"
PROCESSED_DIR = DATA_DIR / "processed"
FLATTENED_DIR = PROCESSED_DIR / "flattened"
NORMALIZED_DIR = PROCESSED_DIR / "normalized"
OVERLAY_DIR = PROCESSED_DIR / "overlays"
PROFILE_DIR = DATA_DIR / "imagej_profiles"
EXTRACTED_DIR = DATA_DIR / "extracted_profiles"

for directory in [
    FLATTENED_DIR,
    NORMALIZED_DIR,
    OVERLAY_DIR,
    PROFILE_DIR,
    EXTRACTED_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)

Project root: c:\Users\u0667281\Desktop\JMA\barcoding-amd
Raw data: c:\Users\u0667281\Desktop\JMA\barcoding-amd\data\heyex\meta


In [10]:
CONFIG = {

    # -------- Scan selection --------
    "scan_index": None,          # None = use n//2

    # -------- Flattening --------
    "flatten": True,
    "flatten_layer": "BM",

    # -------- Intensity normalization --------
    "normalize": True,

    "normalization_method": "none",
    # options:
    # "none"
    # "zscore"
    # "minmax"
    # "percentile"

    "percentile_clip": (1, 99),

    # -------- ImageJ line --------
    "line_depth": 25,
    "line_width": 1,

    # -------- Saving --------
    "save_processed": True,
}

In [5]:
e2e_files = sorted(
    list(RAW_DIR.glob("*.E2E"))
    + list(RAW_DIR.glob("*.e2e"))
)

print(f"Found {len(e2e_files)} E2E file(s)\n")

for i, file_path in enumerate(e2e_files):
    print(f"[{i}] {file_path.name}")
if not e2e_files:
    raise FileNotFoundError(
        f"No .E2E files were found in:\n{RAW_DIR}"
    )

Found 20 E2E file(s)

[0] ea12.E2E
[1] ea12.E2E
[2] ea17.E2E
[3] ea17.E2E
[4] ea23.E2E
[5] ea23.E2E
[6] ea35.E2E
[7] ea35.E2E
[8] ea36.E2E
[9] ea36.E2E
[10] ea41.E2E
[11] ea41.E2E
[12] ea47.E2E
[13] ea47.E2E
[14] ea49.E2E
[15] ea49.E2E
[16] ea8.E2E
[17] ea8.E2E
[18] ea9.E2E
[19] ea9.E2E


In [6]:
test_file = e2e_files[0]
print(f"Loading {test_file.name}")
volume = ep.import_heyex_e2e(test_file)
print("Loaded successfully.")
print(type(volume))

Loading ea12.E2E
Loaded successfully.
<class 'eyepy.core.eyevolume.EyeVolume'>


In [7]:
print(volume)
print("\nAvailable attributes:\n")
print(dir(volume))


Available attributes:

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_ascan_maps', '_bscans', '_data', '_data_par', '_default_localizer', '_default_meta', '_estimate_transform', '_layers', '_plot_bscan_positions', '_plot_bscan_region', '_pos_to_localizer_region', '_raw_data', '_slabs', '_volume_maps', 'add_layer_annotation', 'add_pixel_annotation', 'add_slab_annotation', 'data', 'data_par', 'intensity_transform', 'laterality', 'layers', 'load', 'localizer', 'localizer_transform', 'meta', 'par_algorithm', 'plot', 'remove_layer_annotation', 'remove_pixel_annotation', 'remove_slab_annotation', 'save', 'scale'

In [8]:
print("Number of B-scans:", len(volume))

Number of B-scans: 97


In [11]:
if CONFIG["scan_index"] is None:
    scan_index = len(volume) // 2
else:
    scan_index = CONFIG["scan_index"]

print("Selected scan:", scan_index)
bscan = volume[scan_index]

Selected scan: 48


In [12]:
print(type(bscan))
print(dir(bscan))

<class 'eyepy.core.eyebscan.EyeBscan'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'area_maps', 'data', 'index', 'layers', 'meta', 'plot', 'scale_x', 'scale_y', 'shape', 'size_x', 'size_y', 'slabs', 'volume']


In [ ]:
image = bscan.data
print("Type:", type(image))
print("Shape:", image.shape)
print("Dtype:", image.dtype)
print("Min:", image.min())
print("Max:", image.max())

AttributeError: 'EyeBscan' object has no attribute 'image'

In [ ]:
plt.figure(figsize=(12,6))
plt.imshow(image, cmap="gray")
plt.title(f"Central B-scan ({scan_index})")
plt.axis("off")
plt.show()

In [ ]:
print(dir(bscan))

In [ ]:
layers = bscan.layers
print(type(layers))
print(layers)

In [ ]:
# save_path = PROCESSED_DIR / "original"
# save_path.mkdir(parents=True, exist_ok=True)
# outfile = save_path / f"{test_file.stem}.tif"
# tifffile.imwrite(outfile, image)
# print(outfile)

## Standardization and Preprocessing

## Intensity Marking

## Downsteam Analysis